In [57]:
# Imports

import spacy
from nrclex import NRCLex
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import os
import pandas as pd

In [38]:
# Helper functions

analyzer = SentimentIntensityAnalyzer()

def calculate_agency(doc):
    agency_score = 0
    total_tokens = len(doc)

    for token in doc:
        if token.pos_ == "VERB" and token.dep_ == "ROOT":
            if token.dep_ not in ("auxpass",):
                agency_score += 1  

        if token.text.lower() in ("i", "we") and token.dep_ == "nsubj":
            agency_score += 1

        if token.dep_ == "auxpass":
            agency_score -= 0.5

    if total_tokens == 0:
        return 0
    raw = agency_score / total_tokens
    return max(0.0, min(1.0, (raw + 0.5))) 





def calculate_diminishment(doc):
    diminishment_score = 0

    for sent in doc.sents:
        scores = analyzer.polarity_scores(sent.text)
        
        if scores['neg'] > 0.2:
            has_target = any(
                ent.label_ in ("PERSON", "ORG", "NORP")
                for ent in sent.ents
            )
            if has_target:
                diminishment_score += scores['neg'] * 2  
            else:
                diminishment_score += scores['neg'] 

        for token in sent:
            if token.pos_ == "VERB" and token.dep_ == "ROOT":
                if token.morph.get("Mood") == ["Imp"]:
                    diminishment_score += 0.3

    num_sents = len(list(doc.sents))
    if num_sents == 0:
        return 0
    return max(0.0, min(1.0, diminishment_score / num_sents))

In [53]:
# Assess dominance strategy

def assess_dominance_strategy(text):
    doc = nlp(text)
    
    emotion_data = NRCLex(text)
    emotion_data.load_raw_text(text) 
    
    frequencies = emotion_data.affect_frequencies
    
    authority = frequencies.get('anticipation', 0)
    harm = frequencies.get('fear', 0)

    return {
        "agency_score": round(calculate_agency(doc), 4),
        "diminishment_score": round(calculate_diminishment(doc), 4),
        "dominance_score": round(frequencies.get('disgust', 0), 4),
        "moral_frame": "Authority/Dominance" if authority > harm else "Harm/Victim",
        "making_opponent_small": calculate_diminishment(doc) > 0.3
    }

In [61]:
def read_folder(folder_path):
    results = []
    
    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder_path, filename)
            
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()
            
            result = assess_dominance_strategy(text)
            result["filename"] = filename
            results.append(result)
            
            print(f"\n{'='*50}")
            print(f"FILE: {filename}")
            print(f"{'='*50}")
            print(f"  Agency Score:        {result['agency_score']:.4f}")
            print(f"  Dominance Score:     {result['dominance_score']:.4f}")
            print(f"  Diminishment Score:  {result['diminishment_score']:.4f}")
            print(f"  Moral Frame:         {result['moral_frame']}")
            print(f"  Making Opp. Small:   {result['making_opponent_small']}")
    
    return results  # ← this line is likely missing
    


In [63]:
# Andy Beshear

andy_beshear_results = read_folder("Zara-Ahsan-Thesis-Repository/speeches/andy beshear")



FILE: andy_beshear_final_debate.txt
  Agency Score:        0.5783
  Dominance Score:     0.0912
  Diminishment Score:  0.0628
  Moral Frame:         Authority/Dominance
  Making Opp. Small:   False

FILE: andy_beshear_partisan_winning_speech.txt
  Agency Score:        0.5685
  Dominance Score:     0.0342
  Diminishment Score:  0.0291
  Moral Frame:         Authority/Dominance
  Making Opp. Small:   False

FILE: andy_beshear_tweets.txt
  Agency Score:        0.5386
  Dominance Score:     0.0548
  Diminishment Score:  0.0577
  Moral Frame:         Authority/Dominance
  Making Opp. Small:   False

FILE: Fancy Farms.txt
  Agency Score:        0.5695
  Dominance Score:     0.0735
  Diminishment Score:  0.0513
  Moral Frame:         Authority/Dominance
  Making Opp. Small:   False


In [64]:
def process_all_speakers(speaker_folders: dict):
    """
    speaker_folders: a dictionary of {speaker_name: folder_path}
    """
    all_results = []
    
    for speaker, folder_path in speaker_folders.items():
        print(f"\nProcessing: {speaker}")
        results = read_folder(folder_path)
        for r in results:
            r["speaker"] = speaker
        all_results.extend(results)
    
    df = pd.DataFrame(all_results)
    df = df[["speaker", "filename", "agency_score", "diminishment_score", 
             "dominance_score", "moral_frame", "making_opponent_small"]]
    
    df.to_excel("thesis_results.xlsx", index=False)
    print("\nSaved to thesis_results.xlsx")
    
    return df

In [65]:
speaker_folders = {
    "Andy Beshar": "Zara-Ahsan-Thesis-Repository/speeches/andy beshear",
    "Anthony Brown": "Zara-Ahsan-Thesis-Repository/speeches/Anthony Brown",
    "Charlie Baker": "Zara-Ahsan-Thesis-Repository/speeches/Charlie Baker",
    "Kris Kobach": "Zara-Ahsan-Thesis-Repository/speeches/Kris Kobach",
    "Larry Hogan": "Zara-Ahsan-Thesis-Repository/speeches/Larry Hogan",
    "Laura Kelly": "Zara-Ahsan-Thesis-Repository/speeches/Laura Kelly",
    "Martha Coakley": "Zara-Ahsan-Thesis-Repository/speeches/Martha Coakley",
    "Matt Bevin": "Zara-Ahsan-Thesis-Repository/speeches/matt bevin",
    "Phil Scott": "Zara-Ahsan-Thesis-Repository/speeches/Phil Scott",
    "Sue Minter": "Zara-Ahsan-Thesis-Repository/speeches/Sue Minter"
}

df = process_all_speakers(speaker_folders)
df


Processing: Andy Beshar

FILE: andy_beshear_final_debate.txt
  Agency Score:        0.5783
  Dominance Score:     0.0912
  Diminishment Score:  0.0628
  Moral Frame:         Authority/Dominance
  Making Opp. Small:   False

FILE: andy_beshear_partisan_winning_speech.txt
  Agency Score:        0.5685
  Dominance Score:     0.0342
  Diminishment Score:  0.0291
  Moral Frame:         Authority/Dominance
  Making Opp. Small:   False

FILE: andy_beshear_tweets.txt
  Agency Score:        0.5386
  Dominance Score:     0.0548
  Diminishment Score:  0.0577
  Moral Frame:         Authority/Dominance
  Making Opp. Small:   False

FILE: Fancy Farms.txt
  Agency Score:        0.5695
  Dominance Score:     0.0735
  Diminishment Score:  0.0513
  Moral Frame:         Authority/Dominance
  Making Opp. Small:   False

Processing: Anthony Brown

FILE: Fox interview.txt
  Agency Score:        0.5682
  Dominance Score:     0.0282
  Diminishment Score:  0.0346
  Moral Frame:         Authority/Dominance
  M

,speaker,filename,agency_score,diminishment_score,dominance_score,moral_frame,making_opponent_small
0,Andy Beshar,andy_beshear_final_debate.txt,0.5783,0.0628,0.0912,Authority/Dominance,False
1,Andy Beshar,andy_beshear_partisan_winning_speech.txt,0.5685,0.0291,0.0342,Authority/Dominance,False
2,Andy Beshar,andy_beshear_tweets.txt,0.5386,0.0577,0.0548,Authority/Dominance,False
3,Andy Beshar,Fancy Farms.txt,0.5695,0.0513,0.0735,Authority/Dominance,False
4,Anthony Brown,Fox interview.txt,0.5682,0.0346,0.0282,Authority/Dominance,False
5,Anthony Brown,Oct 7 guber debate.txt,0.5774,0.0255,0.0435,Authority/Dominance,False
6,Charlie Baker,MACDC's 2014 Convention.txt,0.5611,0.0152,0.0117,Authority/Dominance,False
7,Charlie Baker,Oct 21 Gubernatorial debate.txt,0.5631,0.0068,0.0281,Authority/Dominance,False
8,Charlie Baker,"Oct 23, 2014 Massachusetts Gubernatorial Debat...",0.5657,0.0167,0.0251,Authority/Dominance,False
9,Charlie Baker,The Berkshire Eagle.txt,0.6136,0.0000,0.0000,Authority/Dominance,False
